In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [7]:
data = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

In [8]:
!wget $data -O data-week-3.csv 

--2026-10-12 21:27:58--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘data-week-3.csv’

data-week-3.csv     100%[===================>] 272.21K  --.-KB/s    in 0.002s  

2026-10-12 21:27:58 (133 MB/s) - ‘data-week-3.csv’ saved [278746/278746]



In [9]:
raw_df = pd.read_csv('data-week-3.csv')
raw_df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


# Data preparation
- Check if the missing values are presented in the features.
- If there are missing values:
- For categorical features, replace them with 'NA'
- For numerical features, replace with with 0.0

In [11]:
# Inspect missing values before imputation.
missing = raw_df.isna().sum().sort_values(ascending=False)
print("Missing values by column (before imputation):")
print(missing.to_string())

df = raw_df.copy()
categorical_features = df.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_features = df.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [c for c in categorical_features if c != "converted"]
numerical_features = [c for c in numerical_features if c != "converted"]

df[categorical_features] = df[categorical_features].fillna("NA")
df[numerical_features] = df[numerical_features].fillna(0.0)
assert df["converted"].notna().all(), "The target should not contain missing values."
assert int(df.drop(columns=["converted"]).isna().sum().sum()) == 0
print("\nAfter imputation, missing feature values:", int(df.drop(columns=["converted"]).isna().sum().sum()))


Missing values by column (before imputation):
annual_income               369
industry                    240
location                    208
employment_status           194
lead_source                 148
lead_score                   35
number_of_courses_viewed      0
interaction_count             0
converted                     0

After imputation, missing feature values: 0


/tmp/ipykernel_18875/2274213840.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = df.select_dtypes(include=["object", "category"]).columns.tolist()


# Question 1
## What is the most frequent observation (mode) for the column industry?
- NA
- technology
- healthcare
- retail

In [15]:
industry_mode = df["industry"].mode().iloc[0]
industry_counts = df["industry"].value_counts()
print("Industry counts:")
print(industry_counts.to_string())

Industry counts:
industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240


# Question 2
## Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

### What are the two features that have the biggest correlation?
- interaction_count and lead_score
- number_of_courses_viewed and lead_score
- number_of_courses_viewed and interaction_count
- annual_income and interaction_count
- Only consider the pairs above when answering this question.

In [18]:
numeric_feature_names = [c for c in numerical_features if c != "converted"]
corr = df[numeric_feature_names].corr()
print("Correlation matrix:")
print(corr.round(3).to_string())

candidate_pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]
pair_correlations = pd.Series(
    {f"{a} and {b}": corr.loc[a, b] for a, b in candidate_pairs},
    name="correlation",
).sort_values(ascending=False)
print("\nCandidate pair correlations:")
print(pair_correlations.round(4).to_string())


Correlation matrix:
                          annual_income  number_of_courses_viewed  interaction_count  lead_score
annual_income                     1.000                     0.161              0.123       0.229
number_of_courses_viewed          0.161                     1.000              0.722       0.757
interaction_count                 0.123                     0.722              1.000       0.916
lead_score                        0.229                     0.757              0.916       1.000

Candidate pair correlations:
interaction_count and lead_score                  0.9157
number_of_courses_viewed and lead_score           0.7572
number_of_courses_viewed and interaction_count    0.7216
annual_income and interaction_count               0.1228


Split the data
Split the data with these exact calls:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
Make sure that the target value converted is not in your dataframe.

In [19]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train["converted"].to_numpy()
y_val = df_val["converted"].to_numpy()
y_test = df_test["converted"].to_numpy()
X_train = df_train.drop(columns=["converted"])
X_val = df_val.drop(columns=["converted"])
X_test = df_test.drop(columns=["converted"])

feature_names = X_train.columns.tolist()
print(f"Training rows: {len(X_train):,}; validation rows: {len(X_val):,}; test rows: {len(X_test):,}")
print("Target included in feature frames?", "converted" in X_train.columns)
assert "converted" not in X_train.columns and "converted" not in X_val.columns and "converted" not in X_test.columns


Training rows: 3,000; validation rows: 1,000; test rows: 1,000
Target included in feature frames? False


# Question 3
## Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
Round the scores to 2 decimals using round(score, 2).
Which of these variables has the biggest mutual information score?

- industry
- location
- lead_source
- employment_status

In [22]:
mi_scores = {
    col: round(mutual_info_score(X_train[col], y_train), 2)
    for col in categorical_features
}
mi_table = pd.Series(mi_scores, name="mutual information").sort_values(ascending=False)
print(mi_table.to_string())

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00


## Question 4
### Now let's train a logistic regression.
Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
Fit the model on the training dataset.
To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
What accuracy did you get?

- 0.55
- 0.65
- 0.75
- 0.85

In [23]:
def make_model(columns, C=1.0):
    categorical = [c for c in columns if c in categorical_features]
    numerical = [c for c in columns if c in numerical_features]
    transformer = ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical),
            ("numerical", "passthrough", numerical),
        ],
        remainder="drop",
    )
    model = make_pipeline(
        transformer,
        LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=42),
    )
    return model

model = make_model(feature_names, C=1.0)
model.fit(X_train[feature_names], y_train)
val_predictions = model.predict(X_val[feature_names])
accuracy = accuracy_score(y_val, val_predictions)
print(f"Validation accuracy (unrounded): {accuracy:.6f}")
print(f"Validation accuracy (rounded to 2 decimals): {accuracy:.2f}")
assert round(accuracy, 2) == 0.65


Validation accuracy (unrounded): 0.645000
Validation accuracy (rounded to 2 decimals): 0.65


# Question 5
## Let's find the least useful feature using the feature elimination technique.
Train a model using the same features and parameters as in Q4 (without rounding).
Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
For each feature, calculate the difference between the original accuracy and the accuracy without the feature.
Which of following feature has the smallest difference?

- 'lead_source'
- 'number_of_courses_viewed'
- 'interaction_count'
Note: The difference doesn't have to be positive.

In [27]:
elimination_rows = []
for feature in feature_names:
    remaining = [c for c in feature_names if c != feature]
    candidate_model = make_model(remaining, C=1.0)
    candidate_model.fit(X_train[remaining], y_train)
    candidate_accuracy = accuracy_score(y_val, candidate_model.predict(X_val[remaining]))
    elimination_rows.append({
        "excluded_feature": feature,
        "accuracy_without_feature": candidate_accuracy,
        "original_minus_without": accuracy - candidate_accuracy,
    })

elimination = pd.DataFrame(elimination_rows).set_index("excluded_feature")
print(elimination.to_string(float_format=lambda value: f"{value:.6f}"))
choices = ["lead_source", "number_of_courses_viewed", "interaction_count"]
choice_differences = elimination.loc[choices, "original_minus_without"]
print("\nDifferences for the listed options:")
print(choice_differences.to_string(float_format=lambda value: f"{value:.6f}"))
least_useful = choice_differences.abs().idxmin()



                          accuracy_without_feature  original_minus_without
excluded_feature                                                          
lead_source                               0.642000                0.003000
industry                                  0.645000                0.000000
employment_status                         0.645000                0.000000
location                                  0.645000                0.000000
annual_income                             0.724000               -0.079000
number_of_courses_viewed                  0.643000                0.002000
interaction_count                         0.601000                0.044000
lead_score                                0.644000                0.001000

Differences for the listed options:
excluded_feature
lead_source                0.003000
number_of_courses_viewed   0.002000
interaction_count          0.044000


# Question 6
## Now let's train a regularized logistic regression.
Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
Train models using all the features as in Q4.
Calculate the accuracy on the validation dataset and round it to 3 decimal digits.
Which of these C leads to the best accuracy on the validation set?

- 0.000001
- 0.00001
- 0.0001
- 0.001
Note: If there are multiple options, select the smallest C.

In [26]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]
c_rows = []
for c_value in c_values:
    candidate_model = make_model(feature_names, C=c_value)
    candidate_model.fit(X_train[feature_names], y_train)
    candidate_accuracy = accuracy_score(y_val, candidate_model.predict(X_val[feature_names]))
    c_rows.append({"C": c_value, "accuracy": candidate_accuracy, "accuracy_rounded_3": round(candidate_accuracy, 3)})

c_results = pd.DataFrame(c_rows)
print(c_results.to_string(index=False))
best_accuracy = c_results["accuracy"].max()
best_c = c_results.loc[c_results["accuracy"] == best_accuracy, "C"].min()



       C  accuracy  accuracy_rounded_3
0.000001     0.598               0.598
0.000010     0.598               0.598
0.000100     0.613               0.613
0.001000     0.645               0.645
